# Train YOLO26 for Ethos-U55 deployment (Colab)

Fine-tunes YOLO26n on the `ssperson` dataset and saves `best.pt` to Google Drive.
Next steps (export ONNX → INT8 TFLite → Vela) run locally, see the repo README.

**Runtime → Change runtime type → GPU (T4)** before running.

In [ ]:
!nvidia-smi
!pip install -q "ultralytics>=8.3.0" roboflow

## Config

In [ ]:
MODEL = "yolo26n.pt"
IMGSZ = 320          # use the same size when exporting ONNX
EPOCHS = 100
BATCH = 64           # -1 for auto
DRIVE_DIR = "/content/drive/MyDrive/yolo26-ethos-u55"   # trained outputs are saved here

# Roboflow dataset: https://universe.roboflow.com/hanzhou-7mktt/ssperson/dataset/7
RF_WORKSPACE = "hanzhou-7mktt"
RF_PROJECT = "ssperson"
RF_VERSION = 7
RF_FORMAT = "yolo26"

## Dataset
Downloads the dataset from Roboflow. Add your Roboflow API key as a Colab secret named `ROBOFLOW_API_KEY` (key icon in the left sidebar, enable notebook access). Drive is mounted only to save the trained weights.

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

import pathlib
pathlib.Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
version = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION)
dataset = version.download(RF_FORMAT, location="/content/dataset", overwrite=True)
!ls /content/dataset

The Roboflow `data.yaml` uses `../train/images` paths, which don't resolve reliably, so write a yaml with absolute paths.

In [ ]:
import yaml, pathlib
root = pathlib.Path("/content/dataset")
src = yaml.safe_load((root / "data.yaml").read_text())
cfg = {
    "path": str(root),
    "train": "train/images",
    "val": "valid/images",
    "nc": src["nc"],
    "names": src["names"],
}
if (root / "test/images").exists():
    cfg["test"] = "test/images"
DATA = "/content/data.yaml"
pathlib.Path(DATA).write_text(yaml.safe_dump(cfg))
print(pathlib.Path(DATA).read_text())

## Train

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
result = model.train(
    data=DATA,
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    device=0,
    plots=True,
    project="/content/runs",
    name="yolo26_train",
    exist_ok=True,
)
best = pathlib.Path(result.save_dir) / "weights" / "best.pt"
assert best.exists(), f"{best} not found"
best

## Validate

In [ ]:
metrics = YOLO(str(best)).val(data=DATA, imgsz=IMGSZ)
print("mAP50:", metrics.box.map50, " mAP50-95:", metrics.box.map)

## Save to Drive / download

In [ ]:
import shutil
out = pathlib.Path(DRIVE_DIR) / "best.pt"
shutil.copy2(best, out)
shutil.copytree(result.save_dir, f"{DRIVE_DIR}/yolo26_train", dirs_exist_ok=True)
print("Saved", out)

from google.colab import files
files.download(str(best))   # then place at models/pytorch/best.pt in the repo